# Relational Data and SQL

```{contents}
:local:
:depth: 2
```

A text file is useful for a small export. A relational database becomes useful when records must be related, checked, queried, and updated over many program runs. Use a bookstore to connect the data model to familiar business questions: what is in stock, which books sell, and which records belong together?

The SQL below explains the model. Run queries against a disposable database using a SQLite tool, or adapt the command pattern in the next section. These fenced SQL examples are not browser Run C# cells.


## Relational Model

```{index} Relational Model
```


A database stores information so it can outlive one run of a program. A **relational database** organizes that information into tables with rows, columns, keys, and relationships.


```{index} table; database
```
```{index} row; database
```
```{index} column; database
```


### Tables, Rows, and Columns

```{index} Tables, Rows, and Columns
```


A **table** represents one kind of thing: students, products, orders, books, tickets, or appointments. Each **row** is one item in that table. Each **column** stores one attribute of that item.

| StudentId | FirstName | LastName | Major |
| --- | --- | --- | --- |
| 101 | Ana | Kim | IT |
| 102 | Miles | Chen | CS |
| 103 | Jordan | Smith | Data Analytics |

A good table usually has a clear noun as its name and one row per real-world item.


```{index} primary key
```
```{index} foreign key
```


### Primary Keys and Foreign Keys

```{index} Primary Keys and Foreign Keys
```


A **primary key** uniquely identifies one row in a table. `StudentId` is better than `FirstName` because names can repeat.

A **foreign key** stores the primary key from another table. It lets tables refer to each other without copying the same information repeatedly.

| CourseId | CourseName |
| --- | --- |
| 10 | Intro to Programming |
| 20 | Database Systems |

| EnrollmentId | StudentId | CourseId |
| --- | --- | --- |
| 1 | 101 | 10 |
| 2 | 101 | 20 |
| 3 | 102 | 10 |

The `Enrollments` table connects students to courses. This avoids storing course names inside every student row.


```{index} relationship; database
```


### Relationships

```{index} Relationships
```


Common relationships include:

- **One-to-many**: one customer can have many orders.
- **Many-to-many**: many students can take many courses, usually through a linking table such as `Enrollments`.
- **One-to-one**: one user has one profile, when the details are separated for security or organization.

When designing a database, first name the entities, then choose columns, then decide which keys connect the tables.


```{index} schema
```


### A Small Schema

```{index} A Small Schema
```


A **schema** is the planned structure of a database. Here is a small bookstore schema:

```sql
CREATE TABLE Books (
    BookId INTEGER PRIMARY KEY,
    Title TEXT NOT NULL,
    Author TEXT NOT NULL,
    Price REAL NOT NULL
);

CREATE TABLE Sales (
    SaleId INTEGER PRIMARY KEY,
    BookId INTEGER NOT NULL,
    Quantity INTEGER NOT NULL,
    SoldOn TEXT NOT NULL,
    FOREIGN KEY (BookId) REFERENCES Books(BookId)
);
```

`Books` stores facts about each book. `Sales` stores events that happened over time. The foreign key tells us which book each sale refers to.


### Practice

```{index} Practice
```


1. For a library system, list three tables you would create.
2. Choose a primary key for each table.
3. Identify one foreign key relationship between two tables.


## Queries

```{index} Queries
```


**SQL** is the language used to ask questions about relational data. The most common command is `SELECT`, which reads rows from one or more tables.


```{index} SELECT; SQL
```


### Selecting Columns

```{index} Selecting Columns
```


`SELECT` names the columns to return. `FROM` names the table.

```sql
SELECT Title, Author, Price
FROM Books;
```

Use `SELECT *` only for quick exploration. In application code, name the columns so your program depends on an explicit shape.


```{index} WHERE; SQL
```
```{index} ORDER BY; SQL
```


### Filtering and Sorting

```{index} Filtering and Sorting
```


`WHERE` keeps only rows that match a condition. `ORDER BY` controls the result order.

```sql
SELECT Title, Price
FROM Books
WHERE Price < 25.00
ORDER BY Price DESC;
```

The database does the filtering before your C# program receives the rows. That is usually faster and clearer than loading every row and filtering later.


```{index} aggregation; SQL
```
```{index} GROUP BY; SQL
```


### Aggregation

```{index} Aggregation
```


Aggregate functions summarize many rows into one result.

```sql
SELECT COUNT(*) AS BookCount,
       AVG(Price) AS AveragePrice
FROM Books;
```

`GROUP BY` summarizes rows within each group.

```sql
SELECT Author, COUNT(*) AS BooksWritten
FROM Books
GROUP BY Author
ORDER BY BooksWritten DESC;
```


```{index} JOIN; SQL
```


### Joining Tables

```{index} Joining Tables
```


A `JOIN` combines rows from related tables. The `ON` clause says how the keys match.

```sql
SELECT Books.Title,
       Sales.Quantity,
       Sales.SoldOn
FROM Sales
JOIN Books ON Sales.BookId = Books.BookId
ORDER BY Sales.SoldOn;
```

This query reads sale rows and attaches the book title for each sale. Without the join, the sale row would only have a `BookId`.


### Practice

```{index} Practice
```


Using the `Books` and `Sales` tables, write SQL queries that:

1. list all books by one author;
2. show the three most expensive books;
3. count how many sales exist for each book; and
4. list sale dates with book titles and quantities.


## Changing Data Safely

```{index} Changing Data Safely
```


Queries read data. Information systems also need commands that add, modify, and remove rows. These commands should be written carefully because they change persistent state.


```{index} INSERT; SQL
```


### Inserting Rows

```{index} Inserting Rows
```


`INSERT` adds a new row. Name the columns so the command stays readable if the table later gains another column.

```sql
INSERT INTO Books (Title, Author, Price)
VALUES ('Clean Code', 'Robert C. Martin', 39.99);
```

If the primary key is auto-generated, the database fills it in for you.


```{index} UPDATE; SQL
```


### Updating Rows

```{index} Updating Rows
```


`UPDATE` changes existing rows. Almost every `UPDATE` should include a `WHERE` clause.

```sql
UPDATE Books
SET Price = 34.99
WHERE BookId = 1;
```

Without `WHERE`, the command changes every row in the table. That is rarely what you mean.


```{index} DELETE; SQL
```


### Deleting Rows

```{index} Deleting Rows
```


`DELETE` removes rows. It also needs a precise `WHERE` clause.

```sql
DELETE FROM Books
WHERE BookId = 1;
```

Many applications avoid hard deletion for important business records. They add a column such as `IsArchived` instead, so the history remains available.


```{index} parameterized command
```
```{index} SQL injection
```


### Parameterized Commands

```{index} Parameterized Commands
```


Never build SQL by pasting user input into a string.

```csharp
// Unsafe: user text becomes part of the SQL program
string sql = "SELECT * FROM Books WHERE Title = '" + title + "'";
```

Use parameters instead. The SQL text and user data stay separate.

```csharp
string sql = "SELECT * FROM Books WHERE Title = @title";
command.Parameters.AddWithValue("@title", title);
```

Parameterized commands protect against SQL injection and also make values such as dates and numbers easier to pass correctly.


### Practice

```{index} Practice
```


1. Write an `INSERT` command for a new book.
2. Write an `UPDATE` command that changes a book price by `BookId`.
3. Write a `DELETE` command that removes a sale by `SaleId`.
4. Rewrite this unsafe condition with a parameter: `WHERE Author = '" + author + "'`.


## Inventory Constraints and Money

```{index} Inventory Constraints and Money
```


The lab uses `Books(BookId, Title, PriceCents, Stock)` and `Sales(SaleId, BookId, Quantity)`. Prices are integer cents so a value such as 1999 represents 19.99 without binary floating-point rounding. Require nonnegative prices and stock, positive sale quantities, and a valid book for each sale.

```sql
CREATE TABLE Books (
    BookId INTEGER PRIMARY KEY,
    Title TEXT NOT NULL,
    PriceCents INTEGER NOT NULL CHECK (PriceCents >= 0),
    Stock INTEGER NOT NULL CHECK (Stock >= 0)
);
CREATE TABLE Sales (
    SaleId INTEGER PRIMARY KEY,
    BookId INTEGER NOT NULL REFERENCES Books(BookId),
    Quantity INTEGER NOT NULL CHECK (Quantity > 0)
);
```

The earlier `Books` and `Sales` schema illustrates relationships and dates; this smaller lab schema concentrates on stock. Use one schema consistently within a database. SQLite foreign-key checking is enabled explicitly when the application opens its connection.

Predict which constraint rejects each action: a negative stock count, a sale of zero books, and a sale for an unknown book. Then explain why valid individual rows still do not guarantee that a sale and its stock adjustment happen together.


```{rubric} Footnotes
```
[^1]: Use the supplied local teaching database, rather than a production database.
